In [2]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'fpdf2'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



In [1]:
from fpdf2 import FPDF
print("fpdf2 imported successfully")


ModuleNotFoundError: No module named 'fpdf2'

In [1]:
import pandas as pd
from fpdf2 import FPDF
from datetime import datetime

# ── Load your SQL outputs ───────────────────────────────────────────
q1 = pd.read_csv(r'E:\finsight\data\outputs\q1_grade_analysis.csv')
q2 = pd.read_csv(r'E:\finsight\data\outputs\q2_purpose_analysis.csv')
q3 = pd.read_csv(r'E:\finsight\data\outputs\q3_dti_analysis.csv')
q4 = pd.read_csv(r'E:\finsight\data\outputs\q4_yearly_trend.csv')

# ── Calculate KPIs ──────────────────────────────────────────────────
total_loans     = int(q1['loan_count'].sum())
default_rate    = round((q1['loan_count'] * q1['default_rate_pct']).sum() / q1['loan_count'].sum(), 2)
total_exposure  = round(q1['default_exposure_millions'].sum(), 2)
highest_risk    = q1.loc[q1['default_rate_pct'].idxmax(), 'grade']
riskiest_purpose = q2.loc[q2['default_rate_pct'].idxmax(), 'purpose']
worst_year_rate = q4.loc[q4['default_rate_pct'].idxmax(), 'default_rate_pct']
worst_year      = int(q4.loc[q4['default_rate_pct'].idxmax(), 'issue_year'])

print(f"Total loans:      {total_loans:,}")
print(f"Default rate:     {default_rate}%")
print(f"Total exposure:   ${total_exposure}M")
print(f"Highest risk grade: {highest_risk}")
print(f"Riskiest purpose: {riskiest_purpose}")
print(f"Worst vintage:    {worst_year} ({worst_year_rate}%)")

# ── AI narrative (simulated — replace with API call when available) ─
narrative = f"""
EXECUTIVE RISK SUMMARY — {datetime.now().strftime('%B %Y')}

Portfolio Overview:
The FinSight portfolio comprises {total_loans:,} resolved loans with a blended default 
rate of {default_rate}%. Total identified default exposure across all grade tiers 
reaches ${total_exposure}M. Grade {highest_risk} carries the highest default rate, 
representing the most concentrated risk in the book.

Key Risk Segments:
The highest-risk loan purpose is {riskiest_purpose} with a 31.5% default rate — nearly 
double the portfolio average. High-DTI borrowers (>30% DTI) default at 31.2%, 
representing 126,976 accounts where debt servicing capacity is structurally stretched. 
The {worst_year} vintage cohort shows the highest default rate at {worst_year_rate}%, 
indicating loosened underwriting standards during peak growth years.

Recommendation:
Implement tightened origination thresholds for small business and high-DTI applicants. 
Cap new originations in Grade D-G tiers above 30% DTI until default trends stabilise. 
Prioritise vintage-aware risk monitoring for 2016-2018 cohorts which show sustained 
elevated default rates of 25-27%.

Note: This report is generated automatically from SQL analytics outputs.
AI narrative powered by Google Gemini API (configure API key in scripts/auto_report.py).
"""

print("\nNarrative generated successfully")
print(narrative)

# ── Render PDF ──────────────────────────────────────────────────────
pdf = FPDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)

# Header
pdf.set_font('Helvetica', 'B', 18)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 12, 'FinSight - Weekly Risk Report', ln=True)
pdf.set_font('Helvetica', '', 10)
pdf.set_text_color(100, 100, 100)
pdf.cell(0, 6, f"Generated: {datetime.now().strftime('%d %B %Y')} | Lending Club Portfolio Analysis", ln=True)
pdf.ln(6)

# KPI row
pdf.set_fill_color(238, 243, 251)
pdf.set_text_color(26, 74, 138)
pdf.set_font('Helvetica', 'B', 11)
kpis = [
    ('Total Loans', f"{total_loans:,}"),
    ('Default Rate', f"{default_rate}%"),
    ('Exposure ($M)', f"${total_exposure}M"),
    ('Highest Risk', f"Grade {highest_risk}"),
]
for label, val in kpis:
    pdf.cell(45, 8, label, border=1, align='C', fill=True)
pdf.ln()
pdf.set_font('Helvetica', 'B', 14)
pdf.set_text_color(0, 0, 0)
for label, val in kpis:
    pdf.cell(45, 10, val, border=1, align='C')
pdf.ln(14)

# Grade table
pdf.set_font('Helvetica', 'B', 12)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 8, 'Portfolio Quality by Grade', ln=True)
pdf.set_font('Helvetica', 'B', 9)
pdf.set_fill_color(238, 243, 251)
pdf.set_text_color(0, 0, 0)
headers = ['Grade', 'Loans', 'Default Rate %', 'Avg Rate %', 'Exposure $M']
widths  = [20, 35, 35, 35, 35]
for h, w in zip(headers, widths):
    pdf.cell(w, 7, h, border=1, align='C', fill=True)
pdf.ln()
pdf.set_font('Helvetica', '', 9)
for _, row in q1.iterrows():
    pdf.cell(20, 6, str(row['grade']), border=1, align='C')
    pdf.cell(35, 6, f"{int(row['loan_count']):,}", border=1, align='C')
    pdf.cell(35, 6, f"{row['default_rate_pct']}%", border=1, align='C')
    pdf.cell(35, 6, f"{row['avg_interest_rate']}%", border=1, align='C')
    pdf.cell(35, 6, f"${row['default_exposure_millions']}M", border=1, align='C')
    pdf.ln()
pdf.ln(8)

# Narrative
pdf.set_font('Helvetica', 'B', 12)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 8, 'AI-Generated Executive Summary', ln=True)
pdf.set_font('Helvetica', '', 10)
pdf.set_text_color(0, 0, 0)
pdf.multi_cell(0, 6, narrative.strip())

# Save
output_path = r'E:\finsight\reports\weekly_risk_report.pdf'
pdf.output(output_path)
print(f"\n✓ PDF saved to {output_path}")

ModuleNotFoundError: No module named 'fpdf2'

In [7]:
import subprocess
subprocess.run(['pip', 'install', 'fpdf2'])


CompletedProcess(args=['pip', 'install', 'fpdf2'], returncode=0)

In [9]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'fpdf2'])


CompletedProcess(args=['E:\\finsight\\venv26\\Scripts\\python.exe', '-m', 'pip', 'install', 'fpdf2'], returncode=0)

In [2]:
import sys
print(sys.executable)
print(sys.path)


E:\finsight\venv26\Scripts\python.exe
['C:\\Users\\Raunak\\AppData\\Local\\Python\\pythoncore-3.14-64\\python314.zip', 'C:\\Users\\Raunak\\AppData\\Local\\Python\\pythoncore-3.14-64\\DLLs', 'C:\\Users\\Raunak\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib', 'C:\\Users\\Raunak\\AppData\\Local\\Python\\pythoncore-3.14-64', 'E:\\finsight\\venv26', '', 'E:\\finsight\\venv26\\Lib\\site-packages']


In [3]:
import sys
import subprocess

# Force install into the exact Python being used
subprocess.run([sys.executable, '-m', 'pip', 'install', '--force-reinstall', 'fpdf2'], 
               capture_output=False)

# Force add venv site-packages to path
import site
site.addsitedir(r'E:\finsight\venv26\Lib\site-packages')

# Now try import
from fpdf2 import FPDF
print("✓ fpdf2 working")

ModuleNotFoundError: No module named 'fpdf2'

In [4]:
import sys

# Insert venv site-packages at the very beginning of path
sys.path.insert(0, r'E:\finsight\venv26\Lib\site-packages')

from fpdf2 import FPDF
print("✓ fpdf2 working")

ModuleNotFoundError: No module named 'fpdf2'

In [6]:
import os
packages = os.listdir(r'E:\finsight\venv26\Lib\site-packages')
fpdf_related = [p for p in packages if 'fpdf' in p.lower()]
print(fpdf_related)

['fpdf', 'fpdf2-2.8.7.dist-info']


In [7]:
import sys
sys.path.insert(0, r'E:\finsight\venv26\Lib\site-packages')

from fpdf import FPDF
print("✓ fpdf working")

✓ fpdf working


In [8]:
import pandas as pd
from fpdf import FPDF
from datetime import datetime
import sys
sys.path.insert(0, r'E:\finsight\venv26\Lib\site-packages')

# ── Load SQL outputs ────────────────────────────────────────────────
q1 = pd.read_csv(r'E:\finsight\data\outputs\q1_grade_analysis.csv')
q2 = pd.read_csv(r'E:\finsight\data\outputs\q2_purpose_analysis.csv')
q3 = pd.read_csv(r'E:\finsight\data\outputs\q3_dti_analysis.csv')
q4 = pd.read_csv(r'E:\finsight\data\outputs\q4_yearly_trend.csv')

# ── Calculate KPIs ──────────────────────────────────────────────────
total_loans      = int(q1['loan_count'].sum())
default_rate     = round((q1['loan_count'] * q1['default_rate_pct']).sum() / q1['loan_count'].sum(), 2)
total_exposure   = round(q1['default_exposure_millions'].sum(), 2)
highest_risk     = q1.loc[q1['default_rate_pct'].idxmax(), 'grade']
riskiest_purpose = q2.loc[q2['default_rate_pct'].idxmax(), 'purpose']
worst_year_rate  = q4.loc[q4['default_rate_pct'].idxmax(), 'default_rate_pct']
worst_year       = int(q4.loc[q4['default_rate_pct'].idxmax(), 'issue_year'])

print(f"Total loans:        {total_loans:,}")
print(f"Default rate:       {default_rate}%")
print(f"Total exposure:     ${total_exposure}M")
print(f"Highest risk grade: {highest_risk}")
print(f"Riskiest purpose:   {riskiest_purpose}")
print(f"Worst vintage:      {worst_year} ({worst_year_rate}%)")

# ── AI narrative ────────────────────────────────────────────────────
narrative = f"""EXECUTIVE RISK SUMMARY — {datetime.now().strftime('%B %Y')}

Portfolio Overview:
The FinSight portfolio comprises {total_loans:,} resolved loans with a blended default
rate of {default_rate}%. Total identified default exposure across all grade tiers
reaches ${total_exposure}M. Grade {highest_risk} carries the highest default rate,
representing the most concentrated risk in the book.

Key Risk Segments:
The highest-risk loan purpose is {riskiest_purpose} with a 31.5% default rate — nearly
double the portfolio average. High-DTI borrowers (>30% DTI) default at 31.2%,
representing 126,976 accounts where debt servicing capacity is structurally stretched.
The {worst_year} vintage cohort shows the highest default rate at {worst_year_rate}%,
indicating loosened underwriting standards during peak growth years.

Recommendation:
Implement tightened origination thresholds for small business and high-DTI applicants.
Cap new originations in Grade D-G tiers above 30% DTI until default trends stabilise.
Prioritise vintage-aware risk monitoring for 2016-2018 cohorts which show sustained
elevated default rates of 25-27%.

Note: This report is generated automatically from SQL analytics outputs.
AI narrative generation ready — configure API key in scripts/auto_report.py."""

print("\n✓ Narrative generated")

# ── Render PDF ──────────────────────────────────────────────────────
pdf = FPDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)

# Header
pdf.set_font('Helvetica', 'B', 18)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 12, 'FinSight - Weekly Risk Report', ln=True)
pdf.set_font('Helvetica', '', 10)
pdf.set_text_color(100, 100, 100)
pdf.cell(0, 6, f"Generated: {datetime.now().strftime('%d %B %Y')} | Lending Club Portfolio", ln=True)
pdf.ln(6)

# KPI boxes
pdf.set_fill_color(238, 243, 251)
pdf.set_text_color(26, 74, 138)
pdf.set_font('Helvetica', 'B', 10)
kpis = [
    ('Total Loans',    f"{total_loans:,}"),
    ('Default Rate',   f"{default_rate}%"),
    ('Exposure ($M)',  f"${total_exposure}M"),
    ('Highest Risk',   f"Grade {highest_risk}"),
]
for label, val in kpis:
    pdf.cell(45, 8, label, border=1, align='C', fill=True)
pdf.ln()
pdf.set_font('Helvetica', 'B', 13)
pdf.set_text_color(0, 0, 0)
for label, val in kpis:
    pdf.cell(45, 10, val, border=1, align='C')
pdf.ln(14)

# Grade table
pdf.set_font('Helvetica', 'B', 12)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 8, 'Portfolio Quality by Grade', ln=True)
pdf.set_font('Helvetica', 'B', 9)
pdf.set_fill_color(238, 243, 251)
pdf.set_text_color(0, 0, 0)
headers = ['Grade', 'Loans', 'Default Rate %', 'Avg Rate %', 'Exposure $M']
widths  = [20, 35, 35, 35, 35]
for h, w in zip(headers, widths):
    pdf.cell(w, 7, h, border=1, align='C', fill=True)
pdf.ln()
pdf.set_font('Helvetica', '', 9)
for _, row in q1.iterrows():
    pdf.cell(20, 6, str(row['grade']), border=1, align='C')
    pdf.cell(35, 6, f"{int(row['loan_count']):,}", border=1, align='C')
    pdf.cell(35, 6, f"{row['default_rate_pct']}%", border=1, align='C')
    pdf.cell(35, 6, f"{row['avg_interest_rate']}%", border=1, align='C')
    pdf.cell(35, 6, f"${row['default_exposure_millions']}M", border=1, align='C')
    pdf.ln()
pdf.ln(8)

# Narrative
pdf.set_font('Helvetica', 'B', 12)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 8, 'AI-Generated Executive Summary', ln=True)
pdf.set_font('Helvetica', '', 10)
pdf.set_text_color(0, 0, 0)
pdf.multi_cell(0, 6, narrative.strip())

# Save
output_path = r'E:\finsight\reports\weekly_risk_report.pdf'
pdf.output(output_path)
print(f"\n✓ PDF saved to {output_path}")

Total loans:        1,328,284
Default rate:       21.41%
Total exposure:     $4442.22M
Highest risk grade: G
Riskiest purpose:   small_business
Worst vintage:      2017 (26.8%)

✓ Narrative generated


C:\Users\Raunak\AppData\Local\Temp\ipykernel_19092\326208871.py:64: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 12, 'FinSight - Weekly Risk Report', ln=True)
C:\Users\Raunak\AppData\Local\Temp\ipykernel_19092\326208871.py:67: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 6, f"Generated: {datetime.now().strftime('%d %B %Y')} | Lending Club Portfolio", ln=True)
C:\Users\Raunak\AppData\Local\Temp\ipykernel_19092\326208871.py:92: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 8, 'Portfolio Quality by Grade', ln=True)
C:\Users\Raunak\AppData\Local\Temp\ipykernel_19092\326208871.py:114: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.

FPDFUnicodeEncodingException: Character "—" at index 23 in text is outside the range of characters supported by the font used: "helvetica". Please consider using a Unicode font.

In [9]:
# Fix unicode characters for PDF
narrative_clean = narrative.strip().replace('—', '-').replace('\u2014', '-')

In [10]:
pdf.multi_cell(0, 6, narrative.strip())

FPDFUnicodeEncodingException: Character "—" at index 23 in text is outside the range of characters supported by the font used: "helvetica". Please consider using a Unicode font.

In [11]:
pdf.multi_cell(0, 6, narrative.strip()) 
pdf.multi_cell(0, 6, narrative_clean)

FPDFUnicodeEncodingException: Character "—" at index 23 in text is outside the range of characters supported by the font used: "helvetica". Please consider using a Unicode font.

In [12]:
import pandas as pd
from fpdf import FPDF
from datetime import datetime
import sys
sys.path.insert(0, r'E:\finsight\venv26\Lib\site-packages')

# Load SQL outputs
q1 = pd.read_csv(r'E:\finsight\data\outputs\q1_grade_analysis.csv')
q2 = pd.read_csv(r'E:\finsight\data\outputs\q2_purpose_analysis.csv')
q3 = pd.read_csv(r'E:\finsight\data\outputs\q3_dti_analysis.csv')
q4 = pd.read_csv(r'E:\finsight\data\outputs\q4_yearly_trend.csv')

# Calculate KPIs
total_loans      = int(q1['loan_count'].sum())
default_rate     = round((q1['loan_count'] * q1['default_rate_pct']).sum() / q1['loan_count'].sum(), 2)
total_exposure   = round(q1['default_exposure_millions'].sum(), 2)
highest_risk     = q1.loc[q1['default_rate_pct'].idxmax(), 'grade']
riskiest_purpose = q2.loc[q2['default_rate_pct'].idxmax(), 'purpose']
worst_year_rate  = q4.loc[q4['default_rate_pct'].idxmax(), 'default_rate_pct']
worst_year       = int(q4.loc[q4['default_rate_pct'].idxmax(), 'issue_year'])

print(f"Total loans:        {total_loans:,}")
print(f"Default rate:       {default_rate}%")
print(f"Total exposure:     ${total_exposure}M")
print(f"Highest risk grade: {highest_risk}")
print(f"Riskiest purpose:   {riskiest_purpose}")
print(f"Worst vintage:      {worst_year} ({worst_year_rate}%)")

# Narrative — all special characters replaced with ASCII equivalents
narrative = (
    f"EXECUTIVE RISK SUMMARY - {datetime.now().strftime('%B %Y')}\n\n"
    f"Portfolio Overview:\n"
    f"The FinSight portfolio comprises {total_loans:,} resolved loans with a blended "
    f"default rate of {default_rate}%. Total identified default exposure across all "
    f"grade tiers reaches ${total_exposure}M. Grade {highest_risk} carries the highest "
    f"default rate, representing the most concentrated risk in the book.\n\n"
    f"Key Risk Segments:\n"
    f"The highest-risk loan purpose is {riskiest_purpose} with a 31.5% default rate, "
    f"nearly double the portfolio average. High-DTI borrowers (>30% DTI) default at "
    f"31.2%, representing 126,976 accounts where debt servicing capacity is "
    f"structurally stretched. The {worst_year} vintage cohort shows the highest default "
    f"rate at {worst_year_rate}%, indicating loosened underwriting standards during "
    f"peak growth years.\n\n"
    f"Recommendation:\n"
    f"Implement tightened origination thresholds for small business and high-DTI "
    f"applicants. Cap new originations in Grade D-G tiers above 30% DTI until default "
    f"trends stabilise. Prioritise vintage-aware risk monitoring for 2016-2018 cohorts "
    f"which show sustained elevated default rates of 25-27%.\n\n"
    f"Note: This report is generated automatically from SQL analytics outputs. "
    f"AI narrative generation ready - configure API key in scripts/auto_report.py."
)

print("\nNarrative generated successfully")

# Render PDF
pdf = FPDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)

# Header
pdf.set_font('Helvetica', 'B', 18)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 12, 'FinSight - Weekly Risk Report', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)
pdf.set_text_color(100, 100, 100)
pdf.cell(0, 6, f"Generated: {datetime.now().strftime('%d %B %Y')} | Lending Club Portfolio",
         new_x='LMARGIN', new_y='NEXT')
pdf.ln(6)

# KPI boxes
pdf.set_fill_color(238, 243, 251)
pdf.set_text_color(26, 74, 138)
pdf.set_font('Helvetica', 'B', 10)
kpis = [
    ('Total Loans',   f"{total_loans:,}"),
    ('Default Rate',  f"{default_rate}%"),
    ('Exposure ($M)', f"${total_exposure}M"),
    ('Highest Risk',  f"Grade {highest_risk}"),
]
for label, val in kpis:
    pdf.cell(45, 8, label, border=1, align='C', fill=True)
pdf.ln()
pdf.set_font('Helvetica', 'B', 13)
pdf.set_text_color(0, 0, 0)
for label, val in kpis:
    pdf.cell(45, 10, val, border=1, align='C')
pdf.ln(14)

# Grade table
pdf.set_font('Helvetica', 'B', 12)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 8, 'Portfolio Quality by Grade', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', 'B', 9)
pdf.set_fill_color(238, 243, 251)
pdf.set_text_color(0, 0, 0)
headers = ['Grade', 'Loans', 'Default Rate %', 'Avg Rate %', 'Exposure $M']
widths  = [20, 35, 35, 35, 35]
for h, w in zip(headers, widths):
    pdf.cell(w, 7, h, border=1, align='C', fill=True)
pdf.ln()
pdf.set_font('Helvetica', '', 9)
for _, row in q1.iterrows():
    pdf.cell(20, 6, str(row['grade']), border=1, align='C')
    pdf.cell(35, 6, f"{int(row['loan_count']):,}", border=1, align='C')
    pdf.cell(35, 6, f"{row['default_rate_pct']}%", border=1, align='C')
    pdf.cell(35, 6, f"{row['avg_interest_rate']}%", border=1, align='C')
    pdf.cell(35, 6, f"${row['default_exposure_millions']}M", border=1, align='C')
    pdf.ln()
pdf.ln(8)

# Narrative
pdf.set_font('Helvetica', 'B', 12)
pdf.set_text_color(26, 74, 138)
pdf.cell(0, 8, 'AI-Generated Executive Summary', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 10)
pdf.set_text_color(0, 0, 0)
pdf.multi_cell(0, 6, narrative)

# Save
output_path = r'E:\finsight\reports\weekly_risk_report.pdf'
pdf.output(output_path)
print(f"\n✓ PDF saved to {output_path}")

Total loans:        1,328,284
Default rate:       21.41%
Total exposure:     $4442.22M
Highest risk grade: G
Riskiest purpose:   small_business
Worst vintage:      2017 (26.8%)

Narrative generated successfully

✓ PDF saved to E:\finsight\reports\weekly_risk_report.pdf
